# 05 - Prepare the data for Tableau

**Goal:** turn your clean data into three tidy files that Tableau can use straight away, with no complicated setup inside Tableau.

| File | One row is... | What you will use it for |
|---|---|---|
| `jobs_tableau.csv` | one job | city map, salary charts, role and experience breakdowns |
| `skill_grid.csv` | one job x one skill (including a 0 when the job does **not** ask for it) | skill demand charts that react to every filter |
| `skill_pairs.csv` | one pair of skills within a role group | the "skills that appear together" heatmap |

**Why the "grid" file?** If a job does not mention SQL, the normal skills table has no row for it, so Tableau cannot easily work out "% of jobs that ask for SQL".
In the grid every job has a row for every skill with `has_skill` = 1 or 0, so the **average of `has_skill` is exactly the % of jobs asking for that skill**, and it stays correct when you filter by role, city or experience level.

Run the cells from top to bottom. The files are saved in `data/tableau/`.

## Step 1 - Load the data

In [13]:
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

OUT = Path("../data/tableau")
OUT.mkdir(parents=True, exist_ok=True)

jobs = pd.read_csv("../data/clean/jobs_final.csv")
skills = pd.read_csv("../data/clean/job_skills.csv")

print("jobs:", jobs.shape, "| job-skill pairs:", skills.shape)

jobs: (831, 22) | job-skill pairs: (4629, 3)


## Step 2 - Add map coordinates for cities
Tableau recognises many Indian city names by itself, but not "Delhi NCR". So we add approximate latitude and longitude for the main cities (city centres).
Jobs without a named city (or in a small city not listed) get blanks, and Tableau shows them as "unknown" on the map.

In [14]:
CITY_COORDS = {
    "Bengaluru": (12.9716, 77.5946),
    "Hyderabad": (17.3850, 78.4867),
    "Pune": (18.5204, 73.8567),
    "Mumbai": (19.0760, 72.8777),
    "Delhi NCR": (28.6139, 77.2090),
    "Chennai": (13.0827, 80.2707),
    "Kolkata": (22.5726, 88.3639),
    "Ahmedabad": (23.0225, 72.5714),
    "Jaipur": (26.9124, 75.7873),
    "Kochi": (9.9312, 76.2673),
    "Coimbatore": (11.0168, 76.9558),
    "Mohali": (30.7046, 76.7179),
}

jobs["latitude"] = jobs["city"].map(lambda c: CITY_COORDS.get(c, (np.nan, np.nan))[0])
jobs["longitude"] = jobs["city"].map(lambda c: CITY_COORDS.get(c, (np.nan, np.nan))[1])

print("Jobs placed on the map:", int(jobs["latitude"].notna().sum()), "of", len(jobs))
print()
print("Cities with jobs but no coordinates (add them above if you want them on the map):")
print(jobs.loc[jobs["latitude"].isna() & (jobs["city"] != "Not specified"), "city"].value_counts().head(10))

Jobs placed on the map: 607 of 831

Cities with jobs but no coordinates (add them above if you want them on the map):
city
Thiruvananthapuram    5
Chandigarh            4
Gandhinagar           4
Panchkula             2
Calcutta              2
Bhopal                2
Rewari                1
Kormangala            1
Bhubaneshwar          1
Calicut               1
Name: count, dtype: int64


## Step 3 - Make friendlier labels
Tableau filters look nicer with "Yes/No" than with True/False.

In [15]:
jobs["is_core_role"] = jobs["is_core_role"].map({True: "Yes", False: "No"})
jobs["is_intern"] = jobs["is_intern"].map({True: "Yes", False: "No"})

jobs_tableau = jobs[[
    "job_id", "company", "job_title", "role_group", "is_core_role", "is_intern",
    "city", "latitude", "longitude", "work_mode", "experience_level", "exp_min_years",
    "salary_lpa_clean", "salary_suspect", "company_rating", "company_size", "industry", "sector", "skill_count",
]].copy()

jobs_tableau.to_csv(OUT / "jobs_tableau.csv", index=False, encoding="utf-8")
print("Saved jobs_tableau.csv:", jobs_tableau.shape)

Saved jobs_tableau.csv: (831, 19)


## Step 4 - Build the skill grid
We take every job and pair it with every skill, then mark `has_skill` = 1 where the job really asks for that skill.

In [16]:
all_skills = skills[["skill", "category"]].drop_duplicates().sort_values(["category", "skill"]).reset_index(drop=True)

base = jobs[["job_id", "role_group", "is_core_role", "city", "work_mode", "experience_level"]].merge(all_skills, how="cross")

found = skills[["job_id", "skill"]].assign(has_skill=1)
skill_grid = base.merge(found, on=["job_id", "skill"], how="left")
skill_grid["has_skill"] = skill_grid["has_skill"].fillna(0).astype(int)

skill_grid.to_csv(OUT / "skill_grid.csv", index=False, encoding="utf-8")
print("Saved skill_grid.csv:", skill_grid.shape, "=", len(jobs), "jobs x", len(all_skills), "skills")

Saved skill_grid.csv: (21606, 9) = 831 jobs x 26 skills


In [17]:
# Check: this should match the numbers you saw earlier (for example SQL about 58.6% of core-role jobs)
core_grid = skill_grid[skill_grid["is_core_role"] == "Yes"]
check = (core_grid.groupby("skill")["has_skill"].mean() * 100).round(1).sort_values(ascending=False)
check.head(8)

skill
Communication         63.7
SQL                   58.6
Python                57.0
Statistics            44.9
Data Visualization    41.3
Problem Solving       37.5
Machine Learning      36.1
Spark/Big Data        28.2
Name: has_skill, dtype: float64

## Step 5 - Skill pairs for the heatmap
For each role group (and for "All core roles") we count how many jobs ask for both skills in a pair. Each pair is stored in both orders (SQL, Python) and (Python, SQL) so the heatmap is symmetrical.
`group_size` tells you how many jobs are in the group, so you can ignore very small groups.

In [18]:
skillsets = skills.groupby("job_id")["skill"].apply(lambda s: sorted(set(s))).to_dict()

def pairs_for(label, job_ids):
    counts = Counter()
    for jid in job_ids:
        for a, b in combinations(skillsets.get(jid, []), 2):
            counts[(a, b)] += 1
            counts[(b, a)] += 1
    n = len(job_ids)
    return [(label, a, b, c, n, round(100 * c / n, 1)) for (a, b), c in counts.items()]

core_jobs = jobs[jobs["is_core_role"] == "Yes"]

rows = pairs_for("All core roles", core_jobs["job_id"].tolist())
for role, grp in core_jobs.groupby("role_group"):
    rows += pairs_for(role, grp["job_id"].tolist())

skill_pairs = pd.DataFrame(rows, columns=["group", "skill_1", "skill_2", "jobs_with_both", "group_size", "pct_of_group_jobs"])
skill_pairs.to_csv(OUT / "skill_pairs.csv", index=False, encoding="utf-8")

print("Saved skill_pairs.csv:", skill_pairs.shape)
skill_pairs[skill_pairs["group"] == "All core roles"].sort_values("jobs_with_both", ascending=False).head(10)

Saved skill_pairs.csv: (2900, 6)


,group,skill_1,skill_2,jobs_with_both,group_size,pct_of_group_jobs
85,All core roles,SQL,Python,305,717,42.5
84,All core roles,Python,SQL,305,717,42.5
14,All core roles,Communication,SQL,285,717,39.7
15,All core roles,SQL,Communication,285,717,39.7
12,All core roles,Communication,Python,280,717,39.1
13,All core roles,Python,Communication,280,717,39.1
87,All core roles,Statistics,Python,242,717,33.8
86,All core roles,Python,Statistics,242,717,33.8
1,All core roles,Data Visualization,Communication,235,717,32.8
0,All core roles,Communication,Data Visualization,235,717,32.8


## Checkpoint
You should now have three files in `data/tableau/`. Send me the printed shapes, the "jobs placed on the map" line and the check table under Step 4.